# 실험 제목
- 담당: 김영빈
- 날짜: 26/09/22
- 목적: AI-Hub 라벨링데이터 중 Training(TL, 8만 건)을 정제해서 RAG용 문서(jsonl)로 만들기

> 끝나면 결과를 `experiments/LOG.md`에 한 줄 남기기

## 1단계. 라이브러리 및 경로 설정, 목표 스키마 불러오기

RAG 문서에 어떤 필드를 텍스트/메타데이터로 쓸지는 팀 회의에서 이미 합의되어 `ml/configs/data.yaml`에 반영되어 있다
(`rag.text_fields`, `rag.meta_fields`, `rag.meta_extra`). 이 노트북은 그 설정을 그대로 불러와서 쓰고,
JSON을 읽어 정제하는 로직 자체는 이 노트북에서 직접 구현한다.

원본은 `data/raw/TL_은행|보험|증권/` (AI-Hub 라벨링데이터 중 Training 8만 건), 결과는 Git에 올라가지 않는
`data/processed/`에 저장한다. 지금은 TL(8만 건)만 다루고, Validation(VL, 2만 건)은 다음 단계에서 valid/test로 추가한다.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import yaml
from tqdm.auto import tqdm


def find_project_root(start_path: Path) -> Path:
    """현재 위치부터 상위 폴더를 확인해 프로젝트 루트를 찾는다."""
    start_path = start_path.resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / ".git").exists() and (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. dontalk 저장소 내부에서 노트북을 실행해 주세요."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ml/configs/data.yaml = 팀이 회의로 합의한 RAG 데이터 구조. 여기서 필드 목록만 읽어오고,
# 실제 정제 로직(JSON 파싱, 필터링, 텍스트 조합)은 이 노트북에서 직접 구현한다.
cfg = yaml.safe_load((PROJECT_ROOT / "ml" / "configs" / "data.yaml").read_text(encoding="utf-8"))
TEXT_FIELDS = cfg["rag"]["text_fields"]   # 하나의 text로 합칠 5개 필드: 요구사항/질문/답변/꼬리질문/종합답변
META_FIELDS = cfg["rag"]["meta_fields"]   # RAG metadata로 쓸 4개 필드: category/consulting_topic/qa_topic/consulting_purpose
# 참고: data.yaml 에는 meta_extra(institution/source_id/qa_id)도 정의돼 있지만,
#       검색에 직접 쓰이지 않고 qa_id는 doc_id와 중복이라 이번 버전에서는 쓰지 않기로 함 (처음 합의한 4개만 사용)

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"원본 폴더: {RAW_DIR}")
print(f"저장 폴더: {PROCESSED_DIR}")
print(f"TEXT_FIELDS: {TEXT_FIELDS}")
print(f"META_FIELDS: {META_FIELDS}")

## 2단계. TL 원본 JSON 탐색

`data/raw` 아래 `TL_`로 시작하는 폴더(Training, AI-Hub가 나눠준 8만 건)만 재귀적으로 탐색한다.
`VL_`(Validation)은 이번 단계에서 다루지 않는다.

In [ ]:
json_files = sorted(
    p for p in RAW_DIR.rglob("*.json")
    if p.is_file() and p.relative_to(RAW_DIR).parts[0].startswith("TL_")
)
if not json_files:
    raise FileNotFoundError(f"TL_ 로 시작하는 JSON 파일을 찾지 못했습니다: {RAW_DIR}")

folder_counts = Counter(p.relative_to(RAW_DIR).parts[0] for p in json_files)
for folder, count in sorted(folder_counts.items()):
    print(f"[{folder}] {count:,}개")
print(f"\n총 TL JSON 파일 개수: {len(json_files):,}개")

## 3단계. 목표 스키마로 추출 + 가벼운 품질 검사

원본 JSON 구조를 그대로 펼치지 않고, RAG/SFT에서 실제로 쓸 단순화된 필드만 바로 추출한다
(`source_id`, `qa_id`, `category`, `consulting_topic`, `qa_topic`, `consulting_purpose`,
`instruction`, `question`, `answer`, `follow_up_question`, `output` 등).

필수 텍스트 필드가 비어있으면 제외 대상으로 집계하고, 품질 이슈(마스킹 문자 ● 과다, 극단적으로 짧은 텍스트)는
**제외하지 않고 건수만 집계**해서 보여준다 — 원본을 임의로 손대지 않고 문제 규모를 먼저 눈으로 확인하기 위함.

In [ ]:
REQUIRED_TEXT_FIELDS = ["instruction", "question", "answer", "output"]
MASK_CHAR = "●"
MASK_RATIO_THRESHOLD = 0.1   # 텍스트의 10% 이상이 마스킹 문자면 플래그
SHORT_TEXT_THRESHOLD = 5     # 5자 미만이면 사실상 빈 값으로 간주해 플래그


def flatten_record(data: dict, path: Path) -> dict:
    src, cons = data.get("source", {}), data.get("consulting", {})
    qa_items = data.get("qa_data")
    if not isinstance(qa_items, list) or len(qa_items) != 1:
        raise ValueError(f"qa_data 형식이 예상과 다릅니다: {path.name}")
    qa = qa_items[0]
    inp = qa.get("input", {})

    return {
        "source_id": src.get("source_id"),
        "qa_id": qa.get("qa_id"),
        "institution": src.get("source_institution"),
        "category": cons.get("consulting_category"),
        "consulting_topic": cons.get("consulting_topic"),
        "qa_topic": qa.get("qa_topic"),
        "consulting_purpose": qa.get("consulting_purpose"),
        "task_category": qa.get("task_category"),
        "consulting_situation": qa.get("consulting_situation"),
        "core_financial_terms": qa.get("core_financial_terms"),
        "consulting_summary": cons.get("consulting_summary"),
        "instruction": qa.get("instruction"),
        "question": inp.get("question"),
        "answer": inp.get("answer"),
        "follow_up_question": inp.get("follow_up_question"),
        "output": qa.get("output"),
        "split": "train",   # TL 전량 = train (VL 은 다음 단계에서 valid/test 로 추가 예정)
    }


records, errors = [], []
for path in tqdm(json_files, desc="JSON 파싱"):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        records.append(flatten_record(data, path))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError, ValueError) as e:
        errors.append({"path": str(path), "error": str(e)})

if errors:
    print(f"파싱 실패 {len(errors)}건:")
    display(pd.DataFrame(errors).head(10))
    raise ValueError("파싱 실패한 파일이 있습니다. 위 목록을 확인하세요.")

df = pd.DataFrame(records)
print(f"파싱 완료: {len(df):,}행")

# 필수 필드 결측/공백 체크 (제외 대상)
blank_mask = pd.Series(False, index=df.index)
for col in REQUIRED_TEXT_FIELDS:
    blank_mask |= df[col].fillna("").astype(str).str.strip().eq("")
print(f"필수 필드가 빈 행: {int(blank_mask.sum()):,}건 (제외 예정)")

# 중복 체크
print(f"중복 qa_id: {int(df['qa_id'].duplicated().sum()):,}건")

# 품질 플래그 (제외하지 않고 집계만)
def mask_ratio(text):
    text = text or ""
    return text.count(MASK_CHAR) / len(text) if text else 0

for col in ["question", "answer", "output"]:
    ratios = df[col].fillna("").map(mask_ratio)
    print(f"[{col}] 마스킹 비율 {MASK_RATIO_THRESHOLD:.0%} 이상: {int((ratios >= MASK_RATIO_THRESHOLD).sum()):,}건")
    lengths = df[col].fillna("").str.len()
    print(f"[{col}] 길이 {SHORT_TEXT_THRESHOLD}자 미만: {int((lengths < SHORT_TEXT_THRESHOLD).sum()):,}건")

## 4단계. 정제된 flat 데이터 저장 (`qa_flat.jsonl`)

필수 필드가 빈 행만 제외하고, 나머지는 그대로 보존한다(품질 플래그는 위에서 건수만 확인, 임의로 지우지 않음).
`data/processed/qa_flat.jsonl`은 이후 RAG 문서 생성뿐 아니라 SFT 데이터 생성에도 재사용할 수 있는 중간 산출물이다.

In [ ]:
clean_df = df.loc[~blank_mask].reset_index(drop=True)
print(f"저장 대상: {len(clean_df):,}행 (제외 {int(blank_mask.sum()):,}행)")

qa_flat_path = PROCESSED_DIR / "qa_flat.jsonl"
clean_df.to_json(qa_flat_path, orient="records", lines=True, force_ascii=False)
print(f"저장 완료: {qa_flat_path} ({qa_flat_path.stat().st_size / 1024**2:,.1f} MB)")

## 5단계. RAG 문서(`rag_documents.jsonl`) 생성

`ml/configs/data.yaml`에 정의된 `text_fields`(요구사항/질문/답변/꼬리질문/종합답변) 5개를 라벨을 붙여 하나의 텍스트로 합치고,
`meta_fields`(category/consulting_topic/qa_topic/consulting_purpose) 4개만 `metadata`로 묶는다.
QA 1건 = 문서 1개(청크)로, 별도 chunking은 하지 않는다.

In [ ]:
LABELS = {
    "instruction": "요구사항", "question": "고객 질문", "answer": "상담사 답변",
    "follow_up_question": "꼬리 질문", "output": "종합 답변",
}

def build_text(row) -> str:
    """5개 텍스트 필드를 "[라벨] 내용" 형식으로 이어붙여 하나의 문자열로 만든다. 이게 실제 임베딩 대상이 된다."""
    parts = []
    for field in TEXT_FIELDS:
        value = row.get(field)
        if value:                      # 값이 없는 필드(예: 꼬리질문이 없는 QA)는 건너뛴다
            parts.append(f"[{LABELS.get(field, field)}] {value}")
    return "\n".join(parts)

rag_docs = []
for _, row in tqdm(clean_df.iterrows(), total=len(clean_df), desc="RAG 문서 생성"):
    rag_docs.append({
        "doc_id": row["qa_id"],                             # 문서 고유 ID(원본 QA ID). DB 적재 시 기본키 겸 중복 방지용
        "text": build_text(row),                             # 임베딩할 텍스트 (5개 필드를 합친 것)
        "metadata": {k: row.get(k) for k in META_FIELDS},   # 검색 필터링에 쓸 메타데이터 (합의된 4개만, institution/source_id/qa_id 제외)
    })

rag_df = pd.DataFrame(rag_docs)
print(f"RAG 문서 {len(rag_df):,}건")
rag_df.head(2)

## 6단계. 저장 및 재검증

저장 후 다시 읽어서 행 수·문서 ID 중복·필수 필드(`text`가 비어있지 않은지)를 확인한다.

In [ ]:
rag_path = PROCESSED_DIR / "rag_documents.jsonl"
with rag_path.open("w", encoding="utf-8") as f:
    for doc in rag_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

saved = [json.loads(line) for line in rag_path.open(encoding="utf-8")]
assert len(saved) == len(rag_docs), "저장 전후 건수가 다릅니다"
assert pd.Series([d["doc_id"] for d in saved]).duplicated().sum() == 0, "doc_id 중복이 있습니다"
empty_text = sum(1 for d in saved if not d["text"].strip())
print(f"저장 완료: {rag_path} ({rag_path.stat().st_size / 1024**2:,.1f} MB)")
print(f"저장 건수: {len(saved):,}건, 빈 text: {empty_text}건")

## 관찰 / 메모
- TL 8만 건(은행 40,000 / 보험 24,000 / 증권 16,000) 전부 정상 파싱, 필수 필드 결측 0건, qa_id 중복 0건
- 품질 플래그(제외 안 하고 집계만): 마스킹 10%+ question 366건 / answer 427건 / output 59건, 5자 미만 question 5건 — 전체 대비 소수라 우선 그대로 두고 진행
- metadata는 처음 합의한 4개(category/consulting_topic/qa_topic/consulting_purpose)만 사용 — institution/source_id/qa_id는 검색에 안 쓰여서 제외 (source_id 다양성 필터는 나중에 필요해지면 재검토)
- 결과: data/processed/qa_flat.jsonl(183MB), data/processed/rag_documents.jsonl(138MB) — 80,000건, 전부 split="train"
- VL 2만 건은 다음 단계(valid/test)에서 이어서 처리 예정
